In [18]:
from langgraph.graph import StateGraph, START, END
from groq import Groq
  # Groq model SDK
from dotenv import load_dotenv
from typing import TypedDict, Annotated,Literal
from pydantic import BaseModel, Field, field_validator
import operator
import os

In [2]:
load_dotenv()

True

In [3]:
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [4]:
class SentimentSchema(BaseModel):

    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review')

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

 


In [40]:
import json, re

def groq_structured_call(prompt: str, schema: type[BaseModel]):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
    )

    raw = response.choices[0].message.content

    # 🔥 Extract JSON safely
    json_text = re.search(r"\{.*\}", raw, re.S).group()
    data = json.loads(json_text)

    return schema(**data)

def groq_text(prompt: str) -> str:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()

In [41]:
def sentiment_model(prompt: str) -> SentimentSchema:
    return groq_structured_call(prompt, SentimentSchema)

def diagnosis_model(prompt: str) -> DiagnosisSchema:
    return groq_structured_call(prompt, DiagnosisSchema)


In [42]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [44]:
def find_sentiment(state: ReviewState):

    prompt = f"""
    Determine the sentiment of the following review.
    Respond ONLY in JSON format:
    {{
        "sentiment": "positive" | "negative"
    }}

    Review:
    {state["review"]}
    """

    output = groq_structured_call(prompt, SentimentSchema)

    return {"sentiment": output.sentiment}
def run_diagnosis(state: ReviewState):

    prompt = f"""
    Diagnose the following negative review.
    Respond ONLY in valid JSON format:
    {{
        "issue_type": string,
        "tone": string,
        "urgency": string
    }}

    Review:
    {state["review"]}
    """

    output = groq_structured_call(prompt, DiagnosisSchema)

    return {"diagnosis": output.model_dump()}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

def positive_response(state: ReviewState):

    prompt = f"""
    Write a warm thank-you message in response to this review:

    "{state['review']}"

    Also kindly ask the user to leave feedback on our website.
    """

    response = groq_text(prompt)

    return {"response": response}
def negative_response(state: ReviewState):

    diagnosis = state["diagnosis"]

    prompt = f"""
    You are a customer support assistant.

    The user had a '{diagnosis["issue_type"]}' issue,
    sounded '{diagnosis["tone"]}',
    and marked urgency as '{diagnosis["urgency"]}'.

    Write an empathetic, helpful resolution message.
    """

    response = groq_text(prompt)

    return {"response": response}





In [45]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [47]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)

ValidationError: 1 validation error for DiagnosisSchema
issue_type
  Input should be 'UX', 'Performance', 'Bug', 'Support' or 'Other' [type=literal_error, input_value='authentication issue', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error